# Starting-platform efficiency analysis

This notebook compares the current platform-scoped and platform-unknown routing paths without launching a new external search. It combines the local job history with a deterministic replay of the professional-search planner for `octaviyao`.

In [1]:
from types import SimpleNamespace
from statistics import mean, median, stdev

from sqlalchemy import func, select

from apps.api.app.core.config import Settings
from apps.api.app.core.db import build_engine, build_session_factory
from apps.api.app.models.entities import JobAttempt, MaigretScanRun, ProviderRun, SearchJob
from apps.api.app.services.professional_search_scheduling import (
    build_adaptive_professional_query_plan,
    derive_professional_name_hypotheses,
)

settings = Settings()
session_factory = build_session_factory(build_engine(settings.database_url))

with session_factory() as session:
    total, scoped, unknown = session.execute(
        select(
            func.count(SearchJob.id),
            func.count(SearchJob.id).filter(SearchJob.seed_platform.is_not(None)),
            func.count(SearchJob.id).filter(SearchJob.seed_platform.is_(None)),
        ).where(SearchJob.job_kind == "footprint_discovery")
    ).one()

print({
    "discovery_jobs": total,
    "platform_scoped_jobs": scoped,
    "platform_unknown_jobs": unknown,
})

{'discovery_jobs': 15, 'platform_scoped_jobs': 15, 'platform_unknown_jobs': 0}


In [2]:
# Current-pipeline octaviyao runs with 20 completed Maigret checks and
# at least three professional provider runs. This is a platform-scoped
# latency baseline, not a platform-vs-unknown A/B.
with session_factory() as session:
    rows = session.execute(
        select(
            SearchJob.id,
            JobAttempt.started_at,
            JobAttempt.finished_at,
        )
        .join(JobAttempt, JobAttempt.job_id == SearchJob.id)
        .where(
            SearchJob.job_kind == "footprint_discovery",
            func.lower(SearchJob.seed_identifier) == "octaviyao",
            SearchJob.seed_platform == "instagram",
            JobAttempt.finished_at.is_not(None),
            select(func.coalesce(func.sum(MaigretScanRun.completed_count), 0))
            .join(ProviderRun, ProviderRun.id == MaigretScanRun.provider_run_id)
            .where(ProviderRun.job_id == SearchJob.id)
            .scalar_subquery() == 20,
            select(func.count(ProviderRun.id))
            .where(
                ProviderRun.job_id == SearchJob.id,
                ProviderRun.provider_id.in_((
                    "exa_people_search_v1",
                    "github_professional_search_v1",
                )),
            )
            .scalar_subquery() >= 3,
        )
        .order_by(SearchJob.accepted_at)
    ).all()

durations = [(finished - started).total_seconds() for _, started, finished in rows]
summary = {
    "runs": len(durations),
    "median_seconds": round(median(durations), 3),
    "mean_seconds": round(mean(durations), 3),
    "sample_sd_seconds": round(stdev(durations), 3),
    "coefficient_of_variation_pct": round(stdev(durations) / mean(durations) * 100, 1),
    "range_seconds": [round(min(durations), 3), round(max(durations), 3)],
}
print(summary)

{'runs': 6, 'median_seconds': 7.196, 'mean_seconds': 7.359, 'sample_sd_seconds': 1.502, 'coefficient_of_variation_pct': 20.4, 'range_seconds': [6.029, 10.12]}


In [3]:
# Replay the same saved octaviyao evidence with and without the Instagram
# seed-platform tie-breaker. No provider call is made in this cell.
job_id = "4864ed0e-9ebf-467e-9775-afe5d150a1fe"

with session_factory() as session:
    saved_job = session.get(SearchJob, job_id)
    comparisons = []
    for label, platform in (("Instagram known", "instagram"), ("Platform unknown", None)):
        replay_job = SimpleNamespace(
            id=saved_job.id,
            seed_platform=platform,
            seed_identifier=saved_job.seed_identifier,
        )
        hypotheses = derive_professional_name_hypotheses(
            session,
            job=replay_job,
            maximum_names=settings.adaptive_professional_search_max_names,
            include_context_anchors=True,
        )
        plans = build_adaptive_professional_query_plan(
            hypotheses,
            root_handle=saved_job.seed_identifier,
            exa_enabled=False,
            github_enabled=True,
            max_queries=settings.adaptive_professional_search_max_queries,
            max_requests=settings.adaptive_professional_search_max_requests,
            max_profiles=settings.adaptive_professional_search_max_profiles,
        )
        comparisons.append({
            "input_mode": label,
            "maigret_sites": 20,
            "maigret_shards": 3,
            "professional_runs": len(plans),
            "planned_name_queries": sum(len(plan.queries) for plan in plans),
            "request_budget": sum(plan.request_budget for plan in plans),
            "result_budget": sum(plan.result_budget for plan in plans),
            "ordered_hypotheses": [item.full_name for item in hypotheses],
            "root_handle_attached_to": hypotheses[plans[0].hypothesis_index].full_name,
        })

for comparison in comparisons:
    print(comparison)

{'input_mode': 'Instagram known', 'maigret_sites': 20, 'maigret_shards': 3, 'professional_runs': 3, 'planned_name_queries': 3, 'request_budget': 12, 'result_budget': 9, 'ordered_hypotheses': ['Raymond Gu', 'Jingyao Gu', 'Octaviya Octaviya'], 'root_handle_attached_to': 'Raymond Gu'}
{'input_mode': 'Platform unknown', 'maigret_sites': 20, 'maigret_shards': 3, 'professional_runs': 3, 'planned_name_queries': 3, 'request_budget': 12, 'result_budget': 9, 'ordered_hypotheses': ['Jingyao Gu', 'Raymond Gu', 'Octaviya Octaviya'], 'root_handle_attached_to': 'Jingyao Gu'}


## Interpretation

Omitting the platform adds **0% catalog probes** and does not change the `octaviyao` professional-search budget in this deterministic replay. It does change the identity tie-breaker: the original handle is attached to a different name hypothesis. The database has no platform-unknown jobs, so this notebook does not claim an observed wall-clock delta.